# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

This dataset follows the [Croissant](https://mlcommons.org/croissant/) schema and is published at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview

List all available record sets in the dataset by their `@id`, along with their fields and corresponding IDs. All references to entities will use their `@id` as required.

In [ ]:
# List all record sets with their info and fields by @id
from collections.abc import Sequence

# Helper function to ensure record sets list
def get_record_sets(metadata):
    # The Croissant spec says recordSet is usually a list or can be singleton
    record_sets = getattr(metadata, 'recordSet', [])
    if isinstance(record_sets, Sequence) and not isinstance(record_sets, str):
        return list(record_sets)
    elif record_sets:
        return [record_sets]
    else:
        # Try hasPart for record sets (some schemas use this)
        parts = getattr(metadata, 'hasPart', [])
        if isinstance(parts, Sequence) and not isinstance(parts, str):
            return [part for part in parts if getattr(part, '@type', None) == 'RecordSet']
        elif parts:
            if getattr(parts, '@type', None) == 'RecordSet':
                return [parts]
        return []

record_sets = get_record_sets(metadata)

if not record_sets:
    # Try to infer from dataset.records if not found in metadata
    print("No explicit recordSet structure found in metadata; scanning via dataset.records().")
    # We will attempt to enumerate available record_sets from the package data
    # If mlcroissant supports .list_record_sets() we would prefer that
    # For now, let's try a common default @id based on Croissant practices
    # This package appears to use only one main tabular RecordSet
    default_record_set_id = "https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd#tabular1"
    record_sets = [default_record_set_id]

print("Available record sets by @id:")
for rs in record_sets:
    # rs can be an object or a string @id
    if hasattr(rs, '@id'):
        rs_id = getattr(rs, '@id', str(rs))
    else:
        rs_id = str(rs)
    print(f"  - {rs_id}")

    fields = getattr(rs, 'field', []) if hasattr(rs, 'field') else []
    # List fields by @id
    if isinstance(fields, dict):
        fields = [fields]
    elif isinstance(fields, str):
        fields = [fields]
    print("    Fields:")
    for field in fields:
        if hasattr(field, '@id'):
            field_id = getattr(field, '@id')
            print(f"      - {field_id}")
        elif isinstance(field, str):
            print(f"      - {field}")

## 3. Data Extraction

Load data from each record set into a DataFrame using the record set and field `@id`. For this dataset, there is one main record set (tabular data), whose `@id` we will use for data extraction.

In [ ]:
# List all records from each available record set by @id
dataframes = {}

# Use the discovered or default record set(s)
record_set_ids = []
for rs in record_sets:
    rs_id = rs if isinstance(rs, str) else getattr(rs, '@id', str(rs))
    record_set_ids.append(rs_id)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} records from record set: {record_set_id}")
    print("Sample columns:", df.columns[:5].tolist())

# Show all columns of the first record set
main_record_set_id = record_set_ids[0]
print("\nColumn names of main record set:")
print(dataframes[main_record_set_id].columns.tolist())

# Display first 5 rows as preview
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's analyze, filter, and transform the main table using its field and column `@id` values. We demonstrate:
- Selecting a numeric field
- Filtering records based on criteria
- Normalizing a numeric column
- Grouping records by a categorical field

**Note**: If the dataset schema includes only one main numeric field, we will use that for demonstration.

In [ ]:
# Set the record set and field IDs for processing
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Identify a numeric column to use (choose a field typical in clinical data, or inspect columns)
print("Available columns:", df.columns.tolist())

# Example: We'll use 'Age_at_Second_Primary' as the numeric field if present, else pick another (adjust as needed)
possible_numeric_fields = ['Age_at_Second_Primary', 'Interval_Months_Between_Cancers', 'Age_at_First_Primary', 'Interval_Years_Between_Cancers', 'Some_Numeric_Field']
numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break
if not numeric_field:
    raise ValueError("Could not find a typical numeric field in columns. Update the script with a valid field name.")

# Set threshold for filtering records
threshold = 60  # (for example, patient age above 60 at second primary)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold} (n={filtered_df.shape[0]}):")
print(filtered_df.head())

# Normalize the numeric field for the filtered records
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, norm_col]].head())

# Now group by a key categorical field; try 'Sex', else any other clustered variable
group_field_candidates = ['Sex', 'MSI_Status', 'Primary_Cancer_Type']
group_field = None
for gf in group_field_candidates:
    if gf in df.columns:
        group_field = gf
        break

if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    print(grouped.head())
else:
    print("No suitable categorical group field found for grouping.")

## 5. Visualization

Visualize data distributions and relationships in the dataset using matplotlib and seaborn.

In [ ]:
# Visualizations: histogram and boxplot of numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.histplot(df[numeric_field], kde=True, bins=15)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot split by group_field (if available)
if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² colorectal cancer survivors dataset leveraging the Croissant metadata. Using `mlcroissant` and pandas, we:
- Examined available record sets, fields, and columns by their `@id`.
- Loaded data for analysis, using proper reference by `@id`.
- Applied filtering and normalization to numeric clinical variables.
- Performed group-wise statistical analysis and visualized key relationships.

This workflow demonstrates reproducible, standards-based medical data exploration. You can extend this notebook to perform in-depth statistical tests, predictive modeling, or integrate additional FAIR datasets.